# Equational Theories - Exploratory Data Analysis

This notebook explores the competition data to identify patterns and inform cheatsheet design.

**Note**: Using synthetic data since real competition data unavailable.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from parsers import parse_equations, parse_problems, validate_equations, validate_problems
from data_models import Property

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data

In [ ]:
# Load equations
equations = parse_equations('../research/data/original/equations.json')
print(f"Loaded {len(equations)} equations")

# Load problems
problems = parse_problems('../research/data/original/train_problems.json')
print(f"Loaded {len(problems)} problems")

## 2. Equation Statistics

In [ ]:
# Count equations by property
property_counts = Counter()
for eq in equations:
    for prop in eq.properties:
        property_counts[prop.value] += 1

print("Equation counts by property:")
for prop, count in property_counts.most_common():
    print(f"  {prop}: {count}")

In [ ]:
# Visualize property distribution
props = list(property_counts.keys())
counts = list(property_counts.values())

plt.figure(figsize=(10, 6))
plt.bar(props, counts)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Property')
plt.ylabel('Count')
plt.title('Equation Distribution by Property')
plt.tight_layout()
plt.savefig('../research/figures/property_distribution.png', dpi=150)
plt.show()

## 3. Problem Analysis

In [ ]:
# Problem difficulty distribution
difficulty_counts = Counter(p.difficulty for p in problems)
print("\nProblem distribution by difficulty:")
for diff, count in difficulty_counts.items():
    print(f"  {diff}: {count}")

In [ ]:
# Answer distribution
answer_counts = Counter(p.answer for p in problems if p.answer is not None)
print("\nAnswer distribution:")
for answer, count in answer_counts.items():
    print(f"  {answer}: {count}")

true_pct = answer_counts[True] / sum(answer_counts.values()) * 100
false_pct = answer_counts[False] / sum(answer_counts.values()) * 100
print(f"\nTrue implications: {true_pct:.1f}%")
print(f"False implications: {false_pct:.1f}%")

## 4. Frequency Analysis

Which equations appear most frequently in problems?

In [ ]:
# Count equation occurrences
eq1_counts = Counter(p.equation_1_id for p in problems)
eq2_counts = Counter(p.equation_2_id for p in problems)
total_counts = eq1_counts + eq2_counts

print("Top 20 most frequent equations:")
for eq_id, count in total_counts.most_common(20):
    eq = next(e for e in equations if e.id == eq_id)
    props = ", ".join([p.value for p in eq.properties])
    print(f"  E{eq_id} ({eq.name}): {count} occurrences [{props}]")

In [ ]:
# Visualize frequency distribution
top_20_ids = [eq_id for eq_id, _ in total_counts.most_common(20)]
top_20_counts = [total_counts[eq_id] for eq_id in top_20_ids]

plt.figure(figsize=(12, 6))
plt.bar(range(len(top_20_ids)), top_20_counts)
plt.xticks(range(len(top_20_ids)), [f"E{id}" for id in top_20_ids], rotation=45)
plt.xlabel('Equation ID')
plt.ylabel('Frequency')
plt.title('Top 20 Equation Frequencies')
plt.tight_layout()
plt.savefig('../research/figures/equation_frequencies.png', dpi=150)
plt.show()

## 5. Key Insights

### For Cheatsheet Design:

1. **Prioritize by frequency**: The top equations should get more space in the cheatsheet
2. **Property clustering**: Equations with similar properties often appear together
3. **Counterexamples needed**: False implications are common ({false_pct:.1f}%)

### Next Steps:

1. Build implication graph (if data available)
2. Identify hub equations
3. Analyze common implication patterns